# Day 3 — fine-tuning experiment: control (A) vs my SNR-weighted loss (B) + ablations (C, D)

**Settings:** Accelerator = **GPU T4 x2** (both GPUs are used: two runs at a time), Internet = ON.
**Save Version → Save & Run All.** About 2.5–3 h. Download **`day3_results.zip`** from Output at the end.

All four runs start from the SAME released Retinexformer LOL-v1 weights, see the SAME batches in the SAME order
(fixed seed), use the SAME learning rate / iterations — only the loss differs (`scripts/finetune.py`).

Evaluation (all on the final weights — never a checkpoint picked by test score):
* LOL-v1 test (main), LOL-v2-syn (generalisation), LIME/DICM/MEF (NIQE): enhanced images are zipped, scored on the Mac
  with the same `evaluate.py` as every other method.
* **NOT LOL-v2-real:** 91 of its 100 test images are LOL-v1 TRAINING images — the runs trained on them.
* ExDark-200 detection: YOLOv8m on THIS GPU for raw, pretrained Retinexformer and A–D, so all rows share one device.

In [ ]:
# 1. GPU check, our repo (scripts), Retinexformer code, packages
import torch, os, subprocess
assert torch.cuda.device_count() >= 1, "NO GPU! Settings -> Accelerator -> GPU T4 x2"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "| torch", torch.__version__)
%cd /kaggle/working
!git clone -q https://github.com/Crazycraftie/low-light-enhancement-btp.git btp
%cd /kaggle/working/btp
!git log --oneline | head -1
!mkdir -p baselines && git clone -q --depth 1 https://github.com/caiyuanhao1998/Retinexformer.git baselines/Retinexformer
!pip install -q gdown einops addict future lmdb natsort yacs yapf thop timm ultralytics

In [ ]:
# 2. Data + weights from Google Drive (retry: gdown is sometimes flaky)
def gdl(fid, out):
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    for _ in range(4):
        if subprocess.call(f"gdown -q {fid} -O {out}", shell=True) == 0 and os.path.exists(out):
            return
    raise RuntimeError("download failed: " + out)
gdl("1tChRwTfqhs-A67QzG8a9Lrx7qKB3m89K", "baselines/Retinexformer/pretrained_weights/LOL_v1.pth")
for fid, name in [("1L-kqSQyrmMueBh_ziWoPFhfsAh50h20H", "lolv1"), ("1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U", "lolv2"),
                  ("1Pc7IbXQTG30J4JsOzprI8GM0tVNC_AuC", "LIME"), ("15hLlQfrTKbm0iPaogvNk0w81J0ahmMHs", "DICM"),
                  ("1h1_jzihBMykbqYKne0w95qk4WfqQ73HJ", "MEF"), ("1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx", "exdark_images"),
                  ("1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i", "exdark_gt")]:
    gdl(fid, f"/kaggle/tmp/{name}.zip")
!unzip -q -o /kaggle/tmp/lolv1.zip -d data/ && unzip -q -o /kaggle/tmp/lolv2.zip -d data/
!mkdir -p data/unpaired && for d in LIME DICM MEF; do unzip -q -o /kaggle/tmp/$d.zip -d data/unpaired/; done
!mkdir -p data/ExDark /kaggle/tmp/exd && unzip -q -o /kaggle/tmp/exdark_images.zip -d /kaggle/tmp/exd && unzip -q -o /kaggle/tmp/exdark_gt.zip -d /kaggle/tmp/exd
!mv /kaggle/tmp/exd/ExDark data/ExDark/images && mv /kaggle/tmp/exd/ExDark_Annno data/ExDark/annotations
!curl -sL https://raw.githubusercontent.com/cs-chan/Exclusively-Dark-Image-Dataset/master/Groundtruth/imageclasslist.txt -o data/ExDark/imageclasslist.txt
!echo "LOLv1 train $(ls data/LOLv1/Train/input | wc -l) (485) test $(ls data/LOLv1/Test/input | wc -l) (15) | v2-syn $(ls data/LOLv2/Synthetic/Test/Low | wc -l) (100) | LIME $(ls data/unpaired/LIME | wc -l) DICM $(ls data/unpaired/DICM | wc -l) MEF $(ls data/unpaired/MEF | wc -l) (10/69/17)"

In [ ]:
# 3. Fine-tune: A+B in parallel (GPU 0 and GPU 1), then C+D. Same settings for all; only --loss differs.
ITERS = 10000          # ~0.25-0.3 s/iter on a T4 -> ~45-50 min per run; written to every config.json
R = "baselines/Retinexformer"
common = (f"--iters {ITERS} --lr 2e-5 --batch 8 --crop 128 --seed 42 --alpha 1.0 --fft_weight 0.05 "
          f"--pretrained {R}/pretrained_weights/LOL_v1.pth --opt {R}/Options/RetinexFormer_LOL_v1.yml "
          f"--data data/LOLv1/Train --repo {R} --log-every 50")
def pair(a, b):
    procs = []
    for gpu, (run, loss) in enumerate([a, b]):
        cmd = f"CUDA_VISIBLE_DEVICES={gpu} python scripts/finetune.py --loss {loss} --out runs/{run} {common} > runs_{run}.log 2>&1"
        procs.append(subprocess.Popen(cmd, shell=True))
    for p in procs:
        p.wait()
    for run, _ in (a, b):
        ok = os.path.exists(f"runs/{run}/final.pth")
        print(run, "OK" if ok else "FAILED"); os.system(f"tail -n 3 runs_{run}.log")
        assert ok, f"{run} failed - see runs_{run}.log above"
os.makedirs("runs", exist_ok=True)
pair(("A_l1", "l1"), ("B_snr", "snr"))
pair(("C_fft", "fft"), ("D_snr_fft", "snr_fft"))

In [ ]:
# 4. Enhance every test set with each run's FINAL weights (+ pretrained weights on ExDark-200 as reference)
RUNS = {"A_l1": "ft_A_l1", "B_snr": "ft_B_snr", "C_fft": "ft_C_fft", "D_snr_fft": "ft_D_snr_fft"}
SETS = {"LOLv1": "data/LOLv1/Test/input", "LOLv2-syn": "data/LOLv2/Synthetic/Test/Low",
        "LIME": "data/unpaired/LIME", "DICM": "data/unpaired/DICM", "MEF": "data/unpaired/MEF"}
# ExDark: rebuild with OUR script, verify the 200 images pixel-by-pixel, link them into one folder
!python scripts/exdark_to_yolo.py 2>&1 | tail -1
import hashlib, numpy as np
from PIL import Image
rows = [l.split("\t") for l in open("results/exdark_subset200.txt").read().splitlines()[1:]]
bad = [s for s, _, h in rows if hashlib.sha1(np.asarray(Image.open(f"data/exdark_yolo/raw/images/{s}.png").convert("RGB")).tobytes()).hexdigest() != h]
print(f"ExDark pixel check: {len(rows) - len(bad)}/200 identical")
os.makedirs("/kaggle/tmp/exdark200", exist_ok=True)
for s, _, _ in rows:
    if not os.path.exists(f"/kaggle/tmp/exdark200/{s}.png"):
        os.symlink(os.path.abspath(f"data/exdark_yolo/raw/images/{s}.png"), f"/kaggle/tmp/exdark200/{s}.png")
SETS["ExDark"] = "/kaggle/tmp/exdark200"

def infer(weights, folder, inp):
    r = subprocess.run(f"python scripts/retinexformer_infer.py --repo {R} --opt {R}/Options/RetinexFormer_LOL_v1.yml "
                       f"--weights {weights} --input {inp} --output results/{folder} --no-speed-log",
                       shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"inference failed: {weights} on {inp}")
for run, folder in RUNS.items():
    for ds, inp in SETS.items():
        infer(f"runs/{run}/final.pth", f"{folder}/{ds}", inp)
    print(run, {ds: len(os.listdir(f"results/{folder}/{ds}")) for ds in SETS})
infer(f"{R}/pretrained_weights/LOL_v1.pth", "rf_pretrained_gpu/ExDark", SETS["ExDark"])

In [ ]:
# 5. Detection on ExDark-200, all on THIS GPU (suffix _200gpu so the names never clash with the Mac's CPU rows)
!python scripts/build_yolo_sets.py --subset results/exdark_subset200.txt --suffix _200gpu --methods rf_pretrained_gpu ft_A_l1 ft_B_snr ft_C_fft ft_D_snr_fft 2>&1 | grep -E "images ->|WARN"
!python scripts/yolo_eval.py --device 0 --sets raw_200gpu rf_pretrained_gpu_200gpu ft_A_l1_200gpu ft_B_snr_200gpu ft_C_fft_200gpu ft_D_snr_fft_200gpu 2>&1 | grep -E "^=== |^   \{|Error|Traceback"

In [ ]:
# 6. Package: run logs/configs/final weights, enhanced test images (not ExDark - scored here already), detection CSVs + figures
!rm -rf results/ft_*/ExDark results/rf_pretrained_gpu
!cd /kaggle/working/btp && zip -q -r /kaggle/working/day3_results.zip runs results/ft_A_l1 results/ft_B_snr results/ft_C_fft results/ft_D_snr_fft results/detection.csv results/detection_per_class.csv results/figures/detection runs_*.log
!ls -la /kaggle/working/day3_results.zip